In [1]:
import mne
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display
from moabb.datasets import BNCI2014_001
import plotly.io as pio

pio.renderers.default = "vscode"




In [2]:
# 2014
participant = 1
dataset = BNCI2014_001()
data = dataset.get_data(subjects=[participant])

session_key = list(data[participant].keys())[0]
run_key = list(data[participant][session_key].keys())[0]
raw = data[participant][session_key][run_key]

events, event_id = mne.events_from_annotations(raw)
epochs = mne.Epochs(raw, events, event_id=event_id,
tmin=-0.2, tmax=1.0, baseline=None, preload=True)

classes = list(event_id.keys())
classe = classes[3]
idx = 0

ep_classe = epochs[classe]
trial = ep_classe[idx].get_data()[0]
times = ep_classe.times
fig = go.Figure()

for ch_idx, ch_name in enumerate(epochs.ch_names):
    fig.add_trace(go.Scatter(x=times, y=trial[ch_idx], mode='lines', name=ch_name))

fig.update_layout(
    title=f"Sujeito {participant} | Classe: {classe} | Trial {idx+1}/{len(ep_classe)}",
    xaxis_title="Tempo (s)",
    yaxis_title="Amplitude (µV)",
    margin=dict(l=10, r=10, t=40, b=10),
    legend_title="Canal"
)
display(fig)





Used Annotations descriptions: [np.str_('feet'), np.str_('left_hand'), np.str_('right_hand'), np.str_('tongue')]
Not setting metadata
48 matching events found
No baseline correction applied
0 projection items activated
Using data from preloaded Raw for 48 events and 301 original time points ...
0 bad epochs dropped


In [5]:
# SELECIONAR CLASSE

participant = 1
dataset = BNCI2014_001() # dataset usado
data = dataset.get_data(subjects=[participant]) 

session_key = list(data[participant].keys())[0]
run_key = list(data[participant][session_key].keys())[0]
raw = data[participant][session_key][run_key]

events, event_id = mne.events_from_annotations(raw) 
epochs = mne.Epochs(raw, events, event_id=event_id,
tmin=-0.2, tmax=1.0, baseline=None, preload=True)

epochs.pick('eeg') # mantem apenas os canais de eeg

classes = list(event_id.keys()) 
idx = 0
ch_names = epochs.ch_names
n_ch = len(ch_names)

fig = go.Figure()
traces_por_classes = []

for c_idx, classe in enumerate(classes):

    ep_classe = epochs[classe] 
    trial = ep_classe[idx].get_data()[0]
    times = ep_classe.times

    inicio = len(fig.data)
    for ch_idx, ch_name in enumerate(ch_names):
        fig.add_trace(go.Scatter(x=times, y=trial[ch_idx], mode='lines', name=ch_name, legendgroup=f"classe_{c_idx}", visible=(c_idx == 0)))
    traces_por_classes.append((inicio, inicio + n_ch)) 

total_traces = len(fig.data)
buttons = []

for c_idx, classe in enumerate(classes):
    visible_mask = [False] * total_traces
    inicio, fim = traces_por_classes[c_idx]
    for i in range(inicio, fim):
        visible_mask[i] = True

    buttons.append(dict(
        label=f"Classe {classe}",
        method="update",
        args=[
            {"visible": visible_mask},
            {"title": f"Sujeito{participant} | Classe: {classe} | Trial {idx+1}"}
        ]
    ))

fig.update_layout(
    updatemenus=[
        dict(
            buttons=buttons,
            direction="down",
            showactive=True,
            x=0.3,
            xanchor="left",
            y=1.15,
            yanchor="top"
        ),
    ],
    title=f"Sujeito {participant} | Classe: {classes[0]} | Trial {idx+1}",
    xaxis_title="Tempo (s)",
    yaxis_title="Amplitude (µV)",
    margin=dict(l=10, r=10, t=60, b=10),
    legend_title="Canal"
)
display(fig)





Used Annotations descriptions: [np.str_('feet'), np.str_('left_hand'), np.str_('right_hand'), np.str_('tongue')]
Not setting metadata
48 matching events found
No baseline correction applied
0 projection items activated
Using data from preloaded Raw for 48 events and 301 original time points ...
0 bad epochs dropped


In [8]:
# SELECIONAR SESSÃO

participant = 1
dataset = BNCI2014_001()
data = dataset.get_data(subjects=[participant])

session_keys = list(data[participant].keys())

classe_alvo = None   
idx = 0               # índice do trial dentro de cada sessão
ch_names = None
n_ch = None

fig = go.Figure()
traces_por_sessao = []

for s_idx, session_key in enumerate(session_keys):
    # pega o primeiro run dessa sessão
    run_key = list(data[participant][session_key].keys())[0]
    raw = data[participant][session_key][run_key]

    events, event_id = mne.events_from_annotations(raw)
    epochs = mne.Epochs(raw, events, event_id=event_id,
                         tmin=-0.2, tmax=1.0, baseline=None, preload=True)
    epochs.pick('eeg')

    if ch_names is None:
        ch_names = epochs.ch_names
        n_ch = len(ch_names)
        classe_alvo = list(event_id.keys())[0]  

    ep_classe = epochs[classe_alvo]
    trial = ep_classe[idx].get_data()[0]
    times = ep_classe.times

    inicio = len(fig.data)
    for ch_idx, ch_name in enumerate(ch_names):
        fig.add_trace(go.Scatter(
            x=times,
            y=trial[ch_idx],
            mode='lines',
            name=ch_name,
            legendgroup=f"sessao_{s_idx}",
            visible=(s_idx == 0)   
        ))
    traces_por_sessao.append((inicio, inicio + n_ch))


total_traces = len(fig.data)
buttons = []

for s_idx, session_key in enumerate(session_keys):
    visible_mask = [False] * total_traces
    inicio, fim = traces_por_sessao[s_idx]
    for i in range(inicio, fim):
        visible_mask[i] = True

    buttons.append(dict(
        label=f"Sessão {session_key}",
        method="update",
        args=[
            {"visible": visible_mask},
            {"title": f"Sujeito {participant} | Sessão: {session_key} | Classe: {classe_alvo} | Trial {idx+1}"}
        ]
    ))

fig.update_layout(
    updatemenus=[
        dict(
            buttons=buttons,
            direction="down",
            showactive=True,
            x=0.35,
            xanchor="left",
            y=1.15,
            yanchor="top"
        ),
    ],
    title=f"Sujeito {participant} | Sessão: {session_keys[0]} | Classe: {classe_alvo} | Trial {idx+1}",
    xaxis_title="Tempo (s)",
    yaxis_title="Amplitude (µV)",
    margin=dict(l=10, r=10, t=60, b=10),
    legend_title="Canal"
)

display(fig)


Used Annotations descriptions: [np.str_('feet'), np.str_('left_hand'), np.str_('right_hand'), np.str_('tongue')]
Not setting metadata
48 matching events found
No baseline correction applied
0 projection items activated
Using data from preloaded Raw for 48 events and 301 original time points ...
0 bad epochs dropped
Used Annotations descriptions: [np.str_('feet'), np.str_('left_hand'), np.str_('right_hand'), np.str_('tongue')]
Not setting metadata
48 matching events found
No baseline correction applied
0 projection items activated
Using data from preloaded Raw for 48 events and 301 original time points ...
0 bad epochs dropped


In [9]:
# SELECIONAR TRIAL

participant = 1
dataset = BNCI2014_001()
data = dataset.get_data(subjects=[participant])

session_key = list(data[participant].keys())[0]
run_key = list(data[participant][session_key].keys())[0]
raw = data[participant][session_key][run_key]

events, event_id = mne.events_from_annotations(raw)
epochs = mne.Epochs(raw, events, event_id=event_id,
                     tmin=-0.2, tmax=1.0, baseline=None, preload=True)
epochs.pick('eeg')


classes = list(event_id.keys())
classe = classes[1] 

ep_classe = epochs[classe]
ch_names = epochs.ch_names
n_ch = len(ch_names)
times = ep_classe.times
n_trials = len(ep_classe)

print(f"Classe '{classe}' tem {n_trials} trials disponíveis (índices 0 a {n_trials-1})")


fig = go.Figure()
traces_por_trial = []

trials_data = ep_classe.get_data()  

for t_idx in range(n_trials):
    trial = trials_data[t_idx]
    inicio = len(fig.data)

    for ch_idx, ch_name in enumerate(ch_names):
        fig.add_trace(go.Scatter(
            x=times,
            y=trial[ch_idx],
            mode='lines',
            name=ch_name,
            legendgroup=f"trial_{t_idx}",
            visible=(t_idx == 0)  
        ))
    traces_por_trial.append((inicio, inicio + n_ch))


total_traces = len(fig.data)
buttons = []

for t_idx in range(n_trials):
    visible_mask = [False] * total_traces
    inicio, fim = traces_por_trial[t_idx]
    for i in range(inicio, fim):
        visible_mask[i] = True

    buttons.append(dict(
        label=f"Trial {t_idx + 1}",
        method="update",
        args=[
            {"visible": visible_mask},
            {"title": f"Sujeito {participant} | Classe: {classe} | Trial {t_idx + 1}/{n_trials}"}
        ]
    ))

fig.update_layout(
    updatemenus=[
        dict(
            buttons=buttons,
            direction="down",
            showactive=True,
            x=0.22,
            xanchor="left",
            y=1.15,
            yanchor="top"
        ),
    ],
    title=f"Sujeito {participant} | Classe: {classe} | Trial 1/{n_trials}",
    xaxis_title="Tempo (s)",
    yaxis_title="Amplitude (µV)",
    margin=dict(l=10, r=10, t=60, b=10),
    legend_title="Canal"
)

display(fig)


Used Annotations descriptions: [np.str_('feet'), np.str_('left_hand'), np.str_('right_hand'), np.str_('tongue')]
Not setting metadata
48 matching events found
No baseline correction applied
0 projection items activated
Using data from preloaded Raw for 48 events and 301 original time points ...
0 bad epochs dropped
Classe 'left_hand' tem 12 trials disponíveis (índices 0 a 11)


In [ ]:
# SELECIONAR PARTICIPANTES

dataset = BNCI2014_001()
data = dataset.get_data() 

participants = list(data.keys())  # lista de IDs dos participantes

classe_alvo = None
idx = 0            # índice do trial usado para todos os participantes
ch_names = None
n_ch = None

fig = go.Figure()
traces_por_participante = []

for p_idx, participant in enumerate(participants):
    # pega a primeira sessão e o primeiro run desse participante
    session_key = list(data[participant].keys())[0]
    run_key = list(data[participant][session_key].keys())[0]
    raw = data[participant][session_key][run_key]

    events, event_id = mne.events_from_annotations(raw)
    epochs = mne.Epochs(raw, events, event_id=event_id,
                         tmin=-0.2, tmax=1.0, baseline=None, preload=True)
    epochs.pick('eeg')

    if ch_names is None:
        ch_names = epochs.ch_names
        n_ch = len(ch_names)
        classe_alvo = list(event_id.keys())[0]  

    ep_classe = epochs[classe_alvo]
    trial = ep_classe[idx].get_data()[0]
    times = ep_classe.times

    inicio = len(fig.data)
    for ch_idx, ch_name in enumerate(ch_names):
        fig.add_trace(go.Scatter(
            x=times,
            y=trial[ch_idx],
            mode='lines',
            name=ch_name,
            legendgroup=f"participante_{p_idx}",
            visible=(p_idx == 0)  
        ))
    traces_por_participante.append((inicio, inicio + n_ch))


total_traces = len(fig.data)
buttons = []

for p_idx, participant in enumerate(participants):
    visible_mask = [False] * total_traces
    inicio, fim = traces_por_participante[p_idx]
    for i in range(inicio, fim):
        visible_mask[i] = True

    buttons.append(dict(
        label=f"Sujeito {participant}",
        method="update",
        args=[
            {"visible": visible_mask},
            {"title": f"Sujeito {participant} | Classe: {classe_alvo} | Trial {idx+1}"}
        ]
    ))

fig.update_layout(
    updatemenus=[
        dict(
            buttons=buttons,
            direction="down",
            showactive=True,
            x=0.17,
            xanchor="left",
            y=1.15,
            yanchor="top"
        ),
    ],
    title=f"Sujeito {participants[0]} | Classe: {classe_alvo} | Trial {idx+1}",
    xaxis_title="Tempo (s)",
    yaxis_title="Amplitude (µV)",
    margin=dict(l=10, r=10, t=60, b=10),
    legend_title="Canal"
)

fig.show()
